In [210]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [211]:
import random
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from helpers import *
from player import *

In [212]:
class HandEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_head, num_layers):
        super(HandEncoder, self).__init__()
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=input_size, nhead=num_head, dim_feedforward=hidden_size, batch_first=True),
            num_layers=num_layers
        )

        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        pad_mask = (x == torch.full((1, 52), -1, dtype=torch.float32, device=x.device)).sum(dim=2)/52  # Create a mask for padded tokens
        print(pad_mask[0], pad_mask.shape, 'hehe2')
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        x = self.fc(x[:, -1, :])  # Use the last token's output for classification
        return x

In [213]:
class ContextEncoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ContextEncoder, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

        self.fc = nn.Linear(output_size, output_size)

    def forward(self, x):
        x = self.mlp(x)
        x = self.fc(x)
        return x

In [214]:
class PreviousEncoder(nn.Module):
    def __init__(self, hand_input_size, hand_hidden_size, hand_output_size, input_size, hidden_size, output_size):
        super(PreviousEncoder, self).__init__()
        self.handencoder1 = HandEncoder(hand_input_size, hand_hidden_size, hand_output_size, num_head=4, num_layers=2)
        self.handencoder2 = HandEncoder(hand_input_size, hand_hidden_size, hand_output_size, num_head=4, num_layers=2)
        self.handencoder3 = HandEncoder(hand_input_size, hand_hidden_size, hand_output_size, num_head=4, num_layers=2)

        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hand_output_size*3+hidden_size, output_size)

    def forward(self, hand1, hand2, hand3, played_cards):
        hand1_encoded = self.handencoder1(hand1)
        hand2_encoded = self.handencoder2(hand2)
        hand3_encoded = self.handencoder3(hand3)

        # pad_mask = (played_cards == torch.full((1, 52), -1, dtype=torch.float32, device=played_cards.device))
        # print(pad_mask[0], pad_mask.shape, 'hehe')
        # tem = (played_cards != torch.full((1, 52), -1, dtype=torch.float32, device=played_cards.device)).sum(dim=2).sum(dim=1)//52
        # print(tem, tem.shape, tem[0], 'hehe')
        lengths = (played_cards != torch.full((1, 52), -1, dtype=torch.float32, device=played_cards.device)).sum(dim=2).sum(dim=1).cpu()//52
        print(lengths, lengths.shape, lengths[0], 'hehe')
        packed_input = pack_padded_sequence(played_cards, lengths=lengths, batch_first=True, enforce_sorted=False)

        x, (h_n, c_n) = self.lstm(packed_input)
        x = self.fc(torch.cat([hand1_encoded, hand2_encoded, hand3_encoded, h_n[-1]], dim=1))  # Use the last hidden state for classification
        return x

In [215]:
class BidActor(nn.Module):
    def __init__(self, hand_dim, hand_hidden_dim, hand_output_dim, hand_head, hand_layer, 
                 context_dim, context_hidden_dim, context_output_dim, 
                 action_dim, hidden_dim):
        super(BidActor, self).__init__()
        self.hand_encoder = HandEncoder(input_size=hand_dim, hidden_size=hand_hidden_dim, output_size=hand_output_dim, num_head=hand_head, num_layers=hand_layer)
        self.context_encoder = ContextEncoder(input_size=context_dim, hidden_size=context_hidden_dim, output_size=context_output_dim)
        # self.hand_encoder = shared_hand_encoder
        # self.context_encoder = shared_context_encoder

        self.mlp = nn.Sequential(
            nn.Linear(hand_output_dim + context_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, state, mask):
        hand_input, context_input = state['hand'], state['context']
        hand_encoded = self.hand_encoder(hand_input)
        context_encoded = self.context_encoder(context_input)
        action = self.mlp(torch.cat([hand_encoded, context_encoded], dim=1))
        action = action.masked_fill(mask == 0, float('-inf'))  # Apply mask to the output
        return action

In [216]:
class PlayActor(nn.Module):
    def __init__(self, hand_dim, hand_hidden_dim, hand_output_dim, hand_head, hand_layer, 
                 context_dim, context_hidden_dim, context_output_dim, 
                 previous_dim, previous_hidden_dim, previous_output_dim, 
                 action_dim, hidden_dim):
        super(PlayActor, self).__init__()
        self.hand_encoder = HandEncoder(input_size=hand_dim, hidden_size=hand_hidden_dim, output_size=hand_output_dim, num_head=hand_head, num_layers=hand_layer)
        self.context_encoder = ContextEncoder(input_size=context_dim, hidden_size=context_hidden_dim, output_size=context_output_dim)
        # self.handen_coder = shared_hand_encoder
        # self.context_encoder = shared_context_encoder
        self.previous_encoder = PreviousEncoder(hand_input_size=hand_dim, hand_hidden_size=hand_hidden_dim, hand_output_size=hand_output_dim, input_size=previous_dim, hidden_size=previous_hidden_dim, output_size=previous_output_dim)

        self.mlp = nn.Sequential(
            nn.Linear(hand_output_dim + context_output_dim + previous_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, state, mask):
        hand_input, context_input, previous_input = state['hand'], state['context'], state['previous']
        hand_encoded = self.hand_encoder(hand_input)
        context_encoded = self.context_encoder(context_input)
        previous_encoded = self.previous_encoder(previous_input['hand1'], previous_input['hand2'], previous_input['hand3'], previous_input['played_cards'])
        action = self.mlp(torch.cat([hand_encoded, context_encoded, previous_encoded], dim=1))
        action = action.masked_fill(mask == 0, float('-inf'))  # Apply mask to the output
        return action

In [217]:
class Critic(nn.Module):
    def __init__(self, hand_dim, hand_hidden_dim, hand_output_dim, hand_head, hand_layer, 
                 context_dim, context_hidden_dim, context_output_dim, 
                 previous_dim, previous_hidden_dim, previous_output_dim, 
                 action_dim, action_hidden_dim, state_dim, hidden_dim):
        super(Critic, self).__init__()
        self.handen_coder = HandEncoder(input_size=hand_dim, hidden_size=hand_hidden_dim, output_size=hand_output_dim, num_head=hand_head, num_layers=hand_layer)
        self.context_encoder = ContextEncoder(input_size=context_dim, hidden_size=context_hidden_dim, output_size=context_output_dim)
        self.previous_encoder = PreviousEncoder(hand_input_size=hand_dim, hand_hidden_size=hand_hidden_dim, hand_output_size=hand_output_dim, input_size=previous_dim, hidden_size=previous_hidden_dim, output_size=previous_output_dim)

        # self.actionencoder = nn.Sequential(
        #     nn.Linear(action_dim, action_hidden_dim),
        #     nn.ReLU(),
        #     nn.Linear(action_hidden_dim, action_hidden_dim),
        #     nn.ReLU(),
        #     nn.Linear(action_hidden_dim, action_hidden_dim),
        #     nn.ReLU(),
        #     nn.Linear(action_hidden_dim, action_hidden_dim)
        # )

        self.mlp = nn.Sequential(
            nn.Linear(hand_output_dim + context_output_dim + previous_output_dim, state_dim),
            nn.ReLU(),
            nn.Linear(state_dim, state_dim),
            nn.ReLU(),
            nn.Linear(state_dim, state_dim),
            nn.ReLU(),
            nn.Linear(state_dim, state_dim),
            nn.ReLU(),
            nn.Linear(state_dim, 1)  # Output Q-value
        )

        # self.lstm = nn.LSTM(input_size=state_dim, hidden_size=hidden_dim, batch_first=True)
        # self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, state):
        hand_input, context_input, previous_input = state['hand'], state['context'], state['previous']
        hand_encoded = self.handen_coder(hand_input)
        context_encoded = self.context_encoder(context_input)
        previous_encoded = self.previous_encoder(previous_input)
        # action_encoded = self.actionencoder(action_input)

        state_encoded = torch.cat([hand_encoded, context_encoded, previous_encoded], dim=1)
        state_encoded = self.mlp(state_encoded)

        q_value = state_encoded.squeeze(-1)  # Remove the last dimension to get a scalar Q-value

        return q_value

In [218]:
HAND_DIMENSION = 52  # Example dimension for hand representation
ENCODING_DIMENSION = 128  # Example dimension for encoding representation
# CONTEXT_DIMENSION = 100  # Example dimension for context representation
# PREVIOUS_DIMENSION = 50  # Example dimension for previous state representation


In [219]:
hand_dim = HAND_DIMENSION
hand_hidden_dim = 128
hand_output_dim = ENCODING_DIMENSION
hand_head = 4
hand_layer = 2

In [220]:
bidding_context_dim = 12
bidding_context_hidden_dim = 128
bidding_context_output_dim = ENCODING_DIMENSION

bidding_action_dim = 14 
bidding_hidden_dim = 128
bidding_model = BidActor(
    hand_dim=hand_dim,
    hand_hidden_dim=hand_hidden_dim,
    hand_output_dim=hand_output_dim,
    hand_head=hand_head,
    hand_layer=hand_layer,
    context_dim=bidding_context_dim,
    context_hidden_dim=bidding_context_hidden_dim,
    context_output_dim=bidding_context_output_dim,
    action_dim=bidding_action_dim,
    hidden_dim=bidding_hidden_dim
)

playing_context_dim = 15
playing_context_hidden_dim = 128
playing_context_output_dim = ENCODING_DIMENSION
playing_previous_dim = 52
playing_previous_hidden_dim = 128
playing_previous_output_dim = ENCODING_DIMENSION
playing_action_dim = 52
playing_hidden_dim = 128
playing_model = PlayActor(
    hand_dim=hand_dim,
    hand_hidden_dim=hand_hidden_dim,
    hand_output_dim=hand_output_dim,
    hand_head=hand_head,
    hand_layer=hand_layer,
    context_dim=playing_context_dim,
    context_hidden_dim=playing_context_hidden_dim,
    context_output_dim=playing_context_output_dim,
    previous_dim=playing_previous_dim,
    previous_hidden_dim=playing_previous_hidden_dim,
    previous_output_dim=playing_previous_output_dim,
    action_dim=playing_action_dim,
    hidden_dim=playing_hidden_dim
)

In [221]:
critic_state_dim = 256  # Example dimension for combined state representation
critic_hidden_dim = 128  # Example hidden dimension for LSTM
critic_model = Critic(
    hand_dim=hand_dim,
    hand_hidden_dim=hand_hidden_dim,
    hand_output_dim=hand_output_dim,
    hand_head=hand_head,
    hand_layer=hand_layer,
    context_dim=playing_context_dim,
    context_hidden_dim=playing_context_hidden_dim,
    context_output_dim=playing_context_output_dim,
    previous_dim=playing_previous_dim,
    previous_hidden_dim=playing_previous_hidden_dim,
    previous_output_dim=playing_previous_output_dim,
    action_dim=playing_action_dim,  # Assuming the critic evaluates playing actions
    action_hidden_dim=128,  # Example hidden dimension for action encoding
    state_dim=critic_state_dim,  # Example dimension for combined state representation
    hidden_dim=critic_hidden_dim  # Example hidden dimension for LSTM
)

In [222]:
if torch.cuda.is_available():
    device = 'cuda'
    print("Using GPU")
else:
    device = 'cpu'
    print("Using CPU")

bidding_model.to(device)
playing_model.to(device)
critic_model.to(device)

print()
print('----Bidding Model Summary--------------------------------------')
print(f'Number of parameters: {sum(p.numel() for p in bidding_model.parameters())}')
print()
print('----Playing Model Summary--------------------------------------')
print(f'Number of parameters: {sum(p.numel() for p in playing_model.parameters())}')
print()
print('----Critic Model Summary---------------------------------------')
print(f'Number of parameters: {sum(p.numel() for p in critic_model.parameters())}')
print()
print('====RL Model Summary===========================================')
print(f'Number of parameters: {sum(p.numel() for p in bidding_model.parameters()) + sum(p.numel() for p in playing_model.parameters()) + sum(p.numel() for p in critic_model.parameters())}')
print()

Using GPU

----Bidding Model Summary--------------------------------------
Number of parameters: 191670

----Playing Model Summary--------------------------------------
Number of parameters: 540884

----Critic Model Summary---------------------------------------
Number of parameters: 748065

====RL Model Summary===========================================
Number of parameters: 1480619



In [ ]:
def train(bidding_model, playing_model, critic_model,
          bidding_actor_criterion, playing_actor_criterion, critic_criterion, 
          bidding_actor_optimizer, playing_actor_optimizer, critic_optimizer, 
          epoch=10, batch_size=32):
    
    for i in range(epoch):
        bidding_model.train()
        playing_model.train()
        critic_model.train()

        bidding_states = []
        playing_states = []

        # Create a deck for each batch
        deck = list(range(52))
        decks = [deck[:] for x in range(batch_size)]

        # Create players for each batch
        bot_positions = random.choices(range(4), k=batch_size)  # Randomly select bot positions for each batch
        players = []
        for j in range(batch_size):
            player = [AIBot() if bot_positions[j] == x else ComputerBot() for x in range(4)]
            players.append(player)

        score = [0] * 4  # Initialize scores for each player
        scores = [score[:] for x in range(batch_size)]  # Create a score list for each batch

        # Loop through each round of the game (13 rounds for a standard deck)
        for j in range(13):

            triumphs = [None] * batch_size  # Initialize triumphs for each batch

            # Shuffle the decks and deal cards to players for each batch
            for k in range(batch_size):
                random.shuffle(decks[k])
                for l in range(4):
                    players[k][l].hand = decks[k][l*(j+1):(l+1)*(j+1)]  # Deal 13 cards to each player
                if j != 12:  # For the last round, we don't need to set a triumph
                    triumphs[k] = decks[k][4*j] // 13
                
            bid = [None] * 4
            bids = [bid[:] for x in range(batch_size)]
            win = [0] * 4  # Initialize wins for each player
            wins = [win[:] for x in range(batch_size)]  # Create a wins list

            # Loop through each player and bot to get their bids for the current round
            bidding_hand_inputs, bidding_context_inputs = [], []
            
            for k in range(batch_size):
                for l in range(4):
                    if isinstance(players[k][l], AIBot):
                        # Collect the state for the AIBot and get their bid in batch later
                        bidding_hand_input = hand_to_tensor(players[k][l].hand, triumph_suit=triumphs[k]).to(device)
                        bidding_hand_inputs.append(bidding_hand_input)
                        bidding_context_input = context_to_tensor(bids[k], scores[k], bot_positions[k], j+1).to(device)
                        bidding_context_inputs.append(bidding_context_input)
                        # print(f'hand_input: {hand_input.shape}, bidding_context_input: {bidding_context_input.shape}')
                        state = {'hand': bidding_hand_input, 'context': bidding_context_input}
                        bidding_states.append(state)
                        break  # Break after finding the AIBot for this player
                    else:
                        # Get the bid from the ComputerBot and store it in the bids list
                        bids[k][l] = players[k][l].bid(j+1)

            # Get the bid actions from the bidding model for all AIBots in the batch
            bidding_hand_inputs = torch.stack(bidding_hand_inputs, dim=0)
            bidding_context_inputs = torch.stack(bidding_context_inputs, dim=0)

            bidding_mask = torch.zeros((batch_size, bidding_action_dim), dtype=torch.bool).to(device)  # Example mask, adjust as needed
            for k in range(batch_size):
                for l in range(j+2):  # Allow bids up to the current round number 
                    bidding_mask[k][l] = True 
                    if bot_positions[k] == (j+3)%4 and sum(x for x in bids[k] if x is not None)+l == j+1:  # If the AIBot is in this position, allow all bids
                        bidding_mask[k][l] = False
            # print(f'hand_inputs: {hand_inputs.shape}, bidding_context_inputs: {bidding_context_inputs.shape}, mask: {mask}')
            AI_bids_analysis = bidding_model({'hand': bidding_hand_inputs, 'context': bidding_context_inputs}, bidding_mask)
            AI_bids_probabilities = torch.softmax(AI_bids_analysis, dim=1).tolist()
            # print(f"AI_analysis: {AI_bids_analysis}")
            # print(f"AI_probabilities: {AI_bids_probabilities}") 
            AI_bids = [random.choices(range(bidding_action_dim), weights=AI_bids_probabilities[x])[0] for x in range(batch_size)]
            
            # print(f"AI Bids: {AI_bids}")

            for k in range(batch_size):
                AI_bidded = False
                for l in range(4):
                    if bids[k][l] is None:
                        if not AI_bidded:
                            bids[k][l] = AI_bids[k]
                            AI_bidded = True
                        else:
                            bids[k][l] = players[k][l].bid(j+1)  # If the AI has already bid, let the ComputerBot bid

                            # Enforce the rule that the sum of bids cannot equal the number of rounds played
                            if l == 3:
                                while sum(bids[k]) == j+1:
                                    bids[k][l] = players[k][l].bid(j+1)
            print(f"Round {j+1} Bids: {bids}")


            # Loop through each player and bot to play their cards for the current round
            played_hands = [[[] for y in range(4)] for x in range(batch_size)]

            playing_states = []
            playing_positions = [(j + 1) % 4 for x in range(batch_size)]  # Determine the playing position for each batch

            for k in range(j+1):
                played_cards = [[] for x in range(batch_size)]
                playing_hand_inputs, playing_context_inputs, previous_inputs = [], [], []
                for l in range(batch_size):
                    for m in range(4):
                        if isinstance(players[l][playing_positions[l]], AIBot):
                            print(players[l][playing_positions[l]].hand)
                            playing_hand_input = hand_to_tensor(players[l][playing_positions[l]].hand, triumph_suit=triumphs[l]).to(device)
                            playing_hand_inputs.append(playing_hand_input)

                            playing_context_input = context_to_tensor(bids[l], scores[l], bot_positions[l], j+1, players[l][playing_positions[l]].bidded, players[l][playing_positions[l]].won).to(device)
                            playing_context_inputs.append(playing_context_input)

                            previous_hand_tensors, previous_card_tensors = previous_to_tensors(played_hands[l], played_cards[l], triumph_suit=triumphs[l], playing_position=playing_positions[l], max_previous_hand_length=k+1)
                            previous_input = {'hand1': previous_hand_tensors[0].to(device), 'hand2': previous_hand_tensors[1].to(device), 'hand3': previous_hand_tensors[2].to(device), 'played_cards': previous_card_tensors.to(device)}
                            previous_inputs.append(previous_input)

                            state = {'hand': playing_hand_input, 'context': playing_context_input, 'previous': previous_input}
                            playing_states.append(state)
                            break
                        else:
                            
                            played_card = players[l][playing_positions[l]].play_card(played_cards[l], triumphs[l])  # Update the players' hands after bidding
                            played_cards[l].append(played_card)
                            played_hands[l][playing_positions[l]].append(played_card)
                        playing_positions[l] = (playing_positions[l] + 1) % 4
            # print(f"Played Hands: {played_hands}")
                playing_hand_inputs = torch.stack(playing_hand_inputs, dim=0)
                playing_context_inputs = torch.stack(playing_context_inputs, dim=0)
                previous_inputs = {'hand1': torch.stack([x['hand1'] for x in previous_inputs], dim=0),
                                   'hand2': torch.stack([x['hand2'] for x in previous_inputs], dim=0),
                                   'hand3': torch.stack([x['hand3'] for x in previous_inputs], dim=0),
                                   'played_cards': torch.stack([x['played_cards'] for x in previous_inputs], dim=0)}

                print(previous_inputs['hand1'].shape, previous_inputs['hand2'].shape, previous_inputs['hand3'].shape, previous_inputs['played_cards'].shape)
                print(previous_inputs['hand1'][0], previous_inputs['hand2'][0], previous_inputs['hand3'][0], previous_inputs['played_cards'][0])

                playing_mask = torch.zeros((batch_size, HAND_DIMENSION), dtype=torch.bool).to(device)  # Example mask, adjust as needed
                for l in range(batch_size):
                    fixed_suit = None
                    if played_cards[l] != []:  # If the AIBot is not the first play
                        if any(x//13==played_cards[l][0]//13 for x in players[l][bot_positions[l]].hand):  # Check if all cards are in the hand
                            fixed_suit = played_cards[l][0] // 13
                    for m in range(HAND_DIMENSION):  # Allow bids up to the current round number 
                        playing_mask[l][m] = False
                        if m in players[l][bot_positions[l]].hand:
                            if m//13 == triumphs[l]:
                                playing_mask[l][m] = True
                            elif fixed_suit is not None and m//13==fixed_suit:
                                playing_mask[l][m] = True
                            elif fixed_suit is None:
                                playing_mask[l][m] = True

                AI_play_analysis = playing_model({'hand': playing_hand_inputs, 'context': playing_context_inputs, 'previous': previous_inputs}, playing_mask)
                AI_play_probabilities = torch.softmax(AI_play_analysis, dim=1).tolist()
                print(f"AI_play_analysis: {AI_play_analysis}")
                print(f"AI_play_probabilities: {AI_play_probabilities}") 
                AI_plays = [random.choices(range(playing_action_dim), weights=AI_play_probabilities[x])[0] for x in range(batch_size)]
                print(f"AI_plays: {AI_plays}")

                # Critic evaluation for the played actions
                critic_q_values = critic_model({'hand': playing_hand_inputs, 'context': playing_context_inputs, 'previous': previous_inputs})
                print(f"Critic Q-values: {critic_q_values}")













                for l in range(batch_size):
                    players[l][bot_positions[l]].hand.remove(AI_plays[l])  # Remove the played card from the AIBot's hand

                    if len(played_cards[l]) < 3:  # If the AIBot is not the last play
                        playing_positions[l] = (playing_positions[l] + 1) % 4
                        print(playing_positions[l], 'playing_positions[l]', played_cards[l], 'played_cards[l]', (j+1)%4, 'starting position')
                        for m in range(4-len(played_cards[l])-1):
                            print('m', m, players[l][playing_positions[l]].hand, 'hand')
                            played_card = players[l][playing_positions[l]].play_card(played_cards[l], triumphs[l])  # Update the players' hands after bidding
                            played_cards[l].append(played_card)
                            played_hands[l][playing_positions[l]].append(played_card)
                            playing_positions[l] = (playing_positions[l] + 1) % 4

                # Record wins based on the played cards and the rules of the game
                for l in range(batch_size):
                    winning_card = find_winning_card(played_cards[l], triumphs[l])  # Determine the winning card based on triumph suit
                    winning_player = (j + 1 + played_cards[l].index(winning_card)) % 4
                    wins[l][winning_player] += 1  # Update the win for the winning player
                    playing_positions[l] = winning_player  # Update the playing position for the next round
    
            for k in range(batch_size):
                for l in range(4):
                    scores[k][l] += calculate_score(bids[k][l], wins[k][l])  # Update the scores based on bids and wins

            print(f"Round {j+1} Wins: {wins}")
            print(f"Round {j+1} Scores: {scores}")

        # Generate random input data for demonstration purposes
        hand_input = torch.randn(batch_size, 10, HAND_DIMENSION).to(device)  # Example sequence length of 10
        playing_context_input = torch.randn(batch_size, CONTEXT_DIMENSION).to(device)
        previous_input = torch.randn(batch_size, 5, PREVIOUS_DIMENSION).to(device)  # Example sequence length of 5

        # Create state dictionaries for bidding and playing models
        bidding_state = {'hand': hand_input, 'context': bidding_context_input}
        playing_state = {'hand': hand_input, 'context': playing_context_input, 'previous': previous_input}

        # Generate random target actions for demonstration purposes
        bidding_target_actions = torch.randint(0, bidding_action_dim, (batch_size,)).to(device)
        playing_target_actions = torch.randint(0, playing_action_dim, (batch_size,)).to(device)

        # Forward pass through the bidding model
        bidding_output_actions = bidding_model(bidding_state)
        bidding_loss = bidding_actor_criterion(bidding_output_actions, bidding_target_actions)

        # Forward pass through the playing model
        playing_output_actions = playing_model(playing_state)
        playing_loss = playing_actor_criterion(playing_output_actions, playing_target_actions)

        # Forward pass through the critic model
        critic_q_values = critic_model(playing_state)
        critic_target_q_values = torch.randn(batch_size).to(device)  # Random target Q-values for demonstration
        critic_loss = critic_criterion(critic_q_values, critic_target_q_values)

        # Backpropagation and optimization for bidding model
        bidding_actor_optimizer.zero_grad()
        bidding_loss.backward()
        bidding_actor_optimizer.step()

        # Backpropagation and optimization for playing model
        playing_actor_optimizer.zero_grad()
        playing_loss.backward()
        playing_actor_optimizer.step()

        # Backpropagation and optimization for critic model
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()

In [224]:
epoch = 1000
learning_rate = 0.001
batch_size = 32
bidding_actor_criterion = nn.CrossEntropyLoss()
playing_actor_criterion = nn.CrossEntropyLoss()
critic_criterion = nn.MSELoss()
bidding_actor_optimizer = torch.optim.Adam(bidding_model.parameters(), lr=learning_rate)
playing_actor_optimizer = torch.optim.Adam(playing_model.parameters(), lr=learning_rate)
critic_optimizer = torch.optim.Adam(critic_model.parameters(), lr=learning_rate)

train(bidding_model, playing_model, critic_model,
      bidding_actor_criterion, playing_actor_criterion, critic_criterion,
      bidding_actor_optimizer, playing_actor_optimizer, critic_optimizer,
      epoch=epoch, batch_size=batch_size)

tensor([0.], device='cuda:0') torch.Size([32, 1]) hehe2
Round 1 Bids: [[0, 1, 1, 0], [0, 1, 0, 1], [1, 0, 0, 1], [0, 1, 0, 1], [1, 0, 0, 1], [0, 1, 1, 1], [1, 0, 0, 1], [0, 1, 0, 1], [0, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0], [1, 0, 1, 1], [1, 0, 1, 1], [1, 0, 1, 1], [0, 1, 1, 0], [1, 1, 0, 1], [1, 1, 0, 1], [0, 0, 0, 0], [0, 0, 1, 1], [0, 1, 1, 0], [1, 1, 1, 0], [1, 0, 1, 1], [1, 0, 1, 1], [0, 1, 1, 0], [1, 1, 0, 1], [0, 0, 0, 0], [1, 1, 0, 0], [0, 0, 0, 0], [0, 1, 1, 1], [1, 0, 0, 1], [0, 1, 0, 1], [1, 0, 1, 1]]
[12]
[29]
[13]
[30]
[29]
[10]
[45]
[22]
[27]
[17]
[18]
[38]
[39]
[29]
[24]
[32]
[17]
[41]
[4]
[33]
[31]
[10]
[29]
[25]
[35]
[51]
[46]
[1]
[28]
[37]
[1]
[5]
torch.Size([32, 1, 52]) torch.Size([32, 1, 52]) torch.Size([32, 1, 52]) torch.Size([32, 3, 52])
tensor([[-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
         -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
         -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1

NameError: name 'CONTEXT_DIMENSION' is not defined